In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DataType, TimestampType, FloatType
from pyspark.sql import Row

In [0]:
catalog_name = "ecommerce"

#Products

In [0]:
df_products  = spark.table(f"{catalog_name}.silver.slv_products")
df_brands = spark.table(f"{catalog_name}.silver.slv_brands")
df_category = spark.table (f"{catalog_name}.silver.slv_category")

###Create three temporary view

In [0]:
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql("select * from v_products limit 5"))

In [0]:
#Making Sure we're on the right catalog
spark.sql(f"use catalog {catalog_name}")

In [0]:

%sql
--To Build brand x category mapping and trasform to Gold table
--create a CTE (Common Table Expression)

Create Or Replace Table gold.dim_products As

With brands_categories AS (

    SELECT
     b.brand_name,
     b.brand_code,
     c.category_name,
     c.category_code
     FROM v_brands b
     Inner join v_category c
     on 
     b.category_code = c.category_code
)


SELECT
    p.product_id,
    p.sku,
    p.category_code,
    COAlESCE(bc.category_name, 'Not Available') As category_name,
    p.brand_code,
    COALESCE(bc.brand_name, 'Not Available') As brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.file_name,
    p.ingest_timestamp
from v_products p
left join brands_categories bc
on
p.brand_code = bc.brand_code;
